In [3]:
import os

In [4]:
os.listdir('./BRATS 2021 - Task 1/BraTS2021_Training_Data')

['.DS_Store',
 'BraTS2021_00000',
 'BraTS2021_00002',
 'BraTS2021_00003',
 'BraTS2021_00005',
 'BraTS2021_00006',
 'BraTS2021_00008',
 'BraTS2021_00009',
 'BraTS2021_00011',
 'BraTS2021_00012',
 'BraTS2021_00014',
 'BraTS2021_00016',
 'BraTS2021_00017',
 'BraTS2021_00018',
 'BraTS2021_00019',
 'BraTS2021_00020',
 'BraTS2021_00021',
 'BraTS2021_00022',
 'BraTS2021_00024',
 'BraTS2021_00025',
 'BraTS2021_00026',
 'BraTS2021_00028',
 'BraTS2021_00030',
 'BraTS2021_00031',
 'BraTS2021_00032',
 'BraTS2021_00033',
 'BraTS2021_00035',
 'BraTS2021_00036',
 'BraTS2021_00043',
 'BraTS2021_00044',
 'BraTS2021_00045',
 'BraTS2021_00046',
 'BraTS2021_00048',
 'BraTS2021_00049',
 'BraTS2021_00051',
 'BraTS2021_00052',
 'BraTS2021_00053',
 'BraTS2021_00054',
 'BraTS2021_00056',
 'BraTS2021_00058',
 'BraTS2021_00059',
 'BraTS2021_00060',
 'BraTS2021_00061',
 'BraTS2021_00062',
 'BraTS2021_00063',
 'BraTS2021_00064',
 'BraTS2021_00066',
 'BraTS2021_00068',
 'BraTS2021_00070',
 'BraTS2021_00071',
 'BraT

In [5]:
import os
import shutil
from pathlib import Path

# Paths to your downloaded BraTS extracted folders
source_dir = Path('./BRATS 2021 - Task 1/BraTS2021_Training_Data')
target_raw = Path('./nnUNet_raw/Dataset123_BraTS2Modality')

imagestr_dir = target_raw / "imagesTr"
labelstr_dir = target_raw / "labelsTr"
os.makedirs(imagestr_dir, exist_ok=True)
os.makedirs(labelstr_dir, exist_ok=True)

# Loop through all 1,251 clinical patient folders
for patient_folder in source_dir.iterdir():
    if not patient_folder.is_dir():
        continue

    patient_id = patient_folder.name  # e.g., "BraTS2021_00000"

    # Define paths to source files
    t1ce_src = patient_folder / f"{patient_id}_t1ce.nii.gz"
    flair_src = patient_folder / f"{patient_id}_flair.nii.gz"
    seg_src = patient_folder / f"{patient_id}_seg.nii.gz"

    if t1ce_src.exists() and flair_src.exists() and seg_src.exists():
        # Copy and structurally rename for nnU-Net v2 requirements
        shutil.copy(t1ce_src, imagestr_dir / f"{patient_id}_0000.nii.gz")
        shutil.copy(flair_src, imagestr_dir / f"{patient_id}_0001.nii.gz")
        shutil.copy(seg_src, labelstr_dir / f"{patient_id}.nii.gz")

print("Data migration complete!")

Data migration complete!


# Clean BRATS label

Remap the label of BRATS to have a model that just segment GTV:
- NCR (1) -> (1) GTV
- ET (4) -> (1) GTV
- ED (2) -> (0) Background

In [6]:
import os
import nibabel as nib
import numpy as np
from pathlib import Path
from tqdm import tqdm

# Point this to your nnUNet_raw directory labels folder
labels_dir = Path('./nnUNet_raw/Dataset123_BraTS2Modality/labelsTr')

print("Adapting BraTS labels to clean binary GTV masks...")
for mask_path in tqdm(list(labels_dir.glob("*.nii.gz"))):
    # Load the 3D NIfTI image
    img = nib.load(mask_path)
    data = img.get_fdata()

    # Create a clean, empty background matrix of the exact same shape
    binary_gtv = np.zeros_like(data)

    # Map raw BraTS 1 (NCR) and 4 (ET) directly to your new target label: 1
    # Everything else (including ED label 2) remains 0
    binary_gtv[(data == 1) | (data == 4)] = 1

    # Save the modified data back over the file using the original geometry
    new_img = nib.Nifti1Image(binary_gtv.astype(np.uint8), img.affine, img.header)
    nib.save(new_img, mask_path)

print("Finished! Your ground truths are now strictly 0 and 1.")

Adapting BraTS labels to clean binary GTV masks...


100%|██████████| 1251/1251 [01:43<00:00, 12.05it/s]

Finished! Your ground truths are now strictly 0 and 1.
